# 🖐️ 손 랜드마크 검출 (Hand Landmark Detection)

MediaPipe의 **HandLandmarker**를 사용하여 손의 21개 관절(랜드마크)을 검출해 봅니다.

```
┌─────────────────────────────────────────────────────────────┐
│  HandLandmarker 학습 목표                                   │
│                                                             │
│  ① HandLandmarker 설정 (모델 로드)                          │
│  ② 정적 이미지에서 detect() 실행                            │
│  ③ 21개 손 랜드마크 구조 이해                               │
│  ④ drawing_utils로 랜드마크 시각화                          │
│  ⑤ handedness(왼손/오른손) 확인                             │
│  ⑥ 웹캠 실시간 검출 (LIVE_STREAM + 콜백)                    │
│  ⑦ 응용: 손가락 개수 세기                                   │
└─────────────────────────────────────────────────────────────┘
```

| 항목 | 내용 |
|------|------|
| **태스크 클래스** | `mp.tasks.vision.HandLandmarker` |
| **모델 파일** | `models/hand_landmarker.task` |
| **입력** | 이미지 (`mp.Image`) / 비디오 프레임 |
| **출력** | 21개 랜드마크 (x, y, z) + handedness |
| **RunningMode** | `IMAGE` (정적), `LIVE_STREAM` (실시간) |

> MediaPipe 1.0 Tasks API에서는 `HandLandmarker.create_from_options()`로
> 검출기를 생성하고 `detect()` / `detect_for_video()` / `detect_async()`를
> 실행 모드에 맞게 호출합니다.


## 1. 환경 준비

필요한 라이브러리를 임포트하고 MediaPipe 버전을 확인합니다.


In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f'MediaPipe 버전: {mp.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'NumPy: {np.__version__}')

# Tasks API 모듈
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# drawing_utils (랜드마크 시각화)
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands  # 랜드마크 상수/스타일 참고용

print('\n✔ 임포트 완료')


## 2. HandLandmarker 설정

`models/hand_landmarker.task` 모델 파일을 로드하여 HandLandmarker 객체를 만듭니다.

```
┌──────────────────────────────────────────────────────────┐
│  HandLandmarker 생성 절차                                │
│                                                          │
│  BaseOptions(model_asset_path=...)                       │
│        ↓                                                 │
│  HandLandmarkerOptions(base_options=...,                 │
│                        running_mode=IMAGE,               │
│                        num_hands=2)                      │
│        ↓                                                 │
│  vision.HandLandmarker.create_from_options(options)      │
└──────────────────────────────────────────────────────────┘
```

**주요 옵션**
- `num_hands`: 검출할 최대 손 개수 (기본 1)
- `min_hand_detection_confidence`: 손 검출 신뢰도 임계값 (기본 0.5)
- `min_hand_presence_confidence`: 손 존재 신뢰도 (기본 0.5)
- `min_tracking_confidence`: 추적 신뢰도 (LIVE_STREAM에서 사용, 기본 0.5)


In [ ]:
import os

MODEL_PATH = os.path.join('models', 'hand_landmarker.task')
print(f'모델 파일 존재 여부: {os.path.exists(MODEL_PATH)}')

# BaseOptions 설정 — 모델 파일 경로 지정
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

# HandLandmarkerOptions
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,   # 정적 이미지 모드
    num_hands=2,                             # 최대 2개 손 검출
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
)

# 검출기 생성
detector = vision.HandLandmarker.create_from_options(options)
print('✔ HandLandmarker 생성 완료 (IMAGE 모드)')


## 3. 정적 이미지에서 detect()

샘플 손 이미지를 만들어 `detect()`를 실행합니다.
웹캠이 없는 환경을 위해 **합성 이미지**(단색 배경에 손 모양 근사)를
사용하거나, 실제 손 사진 파일이 있으면 `cv2.imread()`로 불러옵니다.

> `mp.Image(mp.ImageFormat.SRGB, np_image)` 로 NumPy 배열을 MediaPipe 이미지로 변환합니다.
> BGR(cv2) → RGB 변환을 먼저 수행하세요.


In [ ]:
# 테스트용 이미지 준비
# 실제 손 이미지가 있으면 아래 주석 해제하여 사용
# img_bgr = cv2.imread('my_hand.jpg')
# img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# 여기서는 mediapipe 패키지에 포함된 샘플을 사용하거나, 합성 이미지로 데모
# 간단한 합성: 검은 배경 (실제 손 이미지로 교체 권장)
img_rgb = np.zeros((480, 640, 3), dtype=np.uint8)
img_rgb[:] = (200, 200, 200)  # 회색 배경

print(f'이미지 shape: {img_rgb.shape}, dtype: {img_rgb.dtype}')

# NumPy → mp.Image 변환
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
print(f'mp.Image: {mp_image}')

# detect() 실행 — RunningMode.IMAGE 모드
result = detector.detect(mp_image)

print(f'\n검출된 손 개수: {len(result.hand_landmarks)}')
print(f'검출된 handedness: {result.handedness}')
print(f'\n✔ detect() 완료')


> 💡 실제 손이 포함된 이미지에서는 `result.hand_landmarks`에 21개 좌표 리스트가
> 손 개수만큼 들어갑니다. 위 합성 이미지에서는 손이 없으므로 빈 리스트가 반환됩니다.
> 아래 셀에서 실제 이미지를 사용하는 예시를 함께 제공합니다.


In [ ]:
# 실제 이미지 파일이 있는 경우의 사용 예시 (템플릿)
def detect_hand_from_file(image_path, detector):
    """이미지 파일에서 손 랜드마크 검출"""
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f'이미지를 불러올 수 없습니다: {image_path}')
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = detector.detect(mp_image)
    return result, img_rgb

# 사용 예 (이미지 경로를 본인 환경에 맞게 수정)
# result, img = detect_hand_from_file('sample_hand.jpg', detector)
# if result and len(result.hand_landmarks) > 0:
#     print(f'검출된 손: {len(result.hand_landmarks)}개')
#     for i, hand in enumerate(result.hand_landmarks):
#         print(f'  손 {i}: {len(hand)}개 랜드마크')
print('detect_hand_from_file() 함수 정의 완료')


## 4. 21개 손 랜드마크 구조

MediaPipe HandLandmarker는 한 손에서 **21개의 랜드마크**를 검출합니다.

```
┌─────────────────────────────────────────────────────────────┐
│                  21개 손 랜드마크                            │
│                                                             │
│   번호   이름            위치                                │
│   ────   ─────────────  ─────────────────────────────       │
│    0     WRIST          손목                                │
│    1     THUMB_CMC      엄지 - 손목 쪽 관절                 │
│    2     THUMB_MCP      엄지 - 손바닥 쪽 관절               │
│    3     THUMB_IP       엄지 - 중간 관절                    │
│    4     THUMB_TIP      엄지 끝                             │
│    5     INDEX_MCP      검지 - 손바닥 쪽 관절               │
│    6     INDEX_PIP      검지 - 첫 번째 관절                 │
│    7     INDEX_DIP      검지 - 두 번째 관절                 │
│    8     INDEX_TIP      검지 끝                             │
│    9     MIDDLE_MCP     중지 - 손바닥 쪽 관절               │
│   10     MIDDLE_PIP     중지 - 첫 번째 관절                 │
│   11     MIDDLE_DIP     중지 - 두 번째 관절                 │
│   12     MIDDLE_TIP     중지 끝                             │
│   13     RING_MCP       약지 - 손바닥 쪽 관절               │
│   14     RING_PIP       약지 - 첫 번째 관절                 │
│   15     RING_DIP       약지 - 두 번째 관절                 │
│   16     RING_TIP       약지 끝                             │
│   17     PINKY_MCP      새끼손가락 - 손바닥 쪽 관절         │
│   18     PINKY_PIP      새끼손가락 - 첫 번째 관절           │
│   19     PINKY_DIP      새끼손가락 - 두 번째 관절           │
│   20     PINKY_TIP      새끼손가락 끝                       │
└─────────────────────────────────────────────────────────────┘
```

**손가락별 관절 (각 4개)**
| 손가락 | 관절 (손목→끝) |
|--------|----------------|
| **엄지 (Thumb)** | CMC → MCP → IP → TIP (1,2,3,4) |
| **검지 (Index)** | MCP → PIP → DIP → TIP (5,6,7,8) |
| **중지 (Middle)** | MCP → PIP → DIP → TIP (9,10,11,12) |
| **약지 (Ring)** | MCP → PIP → DIP → TIP (13,14,15,16) |
| **새끼 (Pinky)** | MCP → PIP → DIP → TIP (17,18,19,20) |

> - **MCP**: Metacarpophalangeal (손가락 밑동, 손바닥 쪽)
> - **PIP**: Proximal Interphalangeal (첫 번째 마디)
> - **DIP**: Distal Interphalangeal (두 번째 마디)
> - **TIP**: 손끝
> - 각 랜드마크는 `(x, y, z)` 좌표를 가집니다. x, y는 정규화된 이미지 좌표(0~1),
>   z는 손목(0) 기준의 상대 깊이입니다(음수 = 카메라에 가까움).


In [ ]:
# 21개 랜드마크 이름 (MediaPipe 표준)
LANDMARK_NAMES = [
    'WRIST',                                            # 0
    'THUMB_CMC', 'THUMB_MCP', 'THUMB_IP', 'THUMB_TIP',  # 1-4
    'INDEX_MCP', 'INDEX_PIP', 'INDEX_DIP', 'INDEX_TIP', # 5-8
    'MIDDLE_MCP', 'MIDDLE_PIP', 'MIDDLE_DIP', 'MIDDLE_TIP', # 9-12
    'RING_MCP', 'RING_PIP', 'RING_DIP', 'RING_TIP',     # 13-16
    'PINKY_MCP', 'PINKY_PIP', 'PINKY_DIP', 'PINKY_TIP', # 17-20
]

print(f'랜드마크 총 개수: {len(LANDMARK_NAMES)}')
for i, name in enumerate(LANDMARK_NAMES):
    finger = '손목' if i == 0 else name.split('_')[0]
    print(f'  {i:2d}: {name:<14s} ({finger})')


## 5. drawing_utils로 랜드마크 시각화

MediaPipe 1.0 Tasks API의 결과는 **기존 `solutions.drawing_utils`와 호환**되지 않는
객체 구조(`list[list[NormalizedLandmark]]`)를 가집니다. 따라서 직접 그리거나,
구 버전 호환 형식(`mp.solution_base.SolutionBase`)으로 변환해야 합니다.

아래는 **직접 OpenCV로 그리는** 범용 함수입니다.


In [ ]:
# MediaPipe 1.0 Tasks 결과를 OpenCV로 직접 그리는 함수
# 21개 랜드마크 연결 관계 (HandLandmarker 표준)
HAND_CONNECTIONS = [
    # 엄지
    (0, 1), (1, 2), (2, 3), (3, 4),
    # 검지
    (0, 5), (5, 6), (6, 7), (7, 8),
    # 중지
    (5, 9), (9, 10), (10, 11), (11, 12),
    # 약지
    (9, 13), (13, 14), (14, 15), (15, 16),
    # 새끼
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20),
]

def draw_hand_landmarks(image, hand_landmarks, connections=HAND_CONNECTIONS,
                        landmark_color=(0, 255, 0), connection_color=(255, 255, 0),
                        thickness=2, radius=4):
    """Tasks API 결과를 이미지에 그리기 (image는 RGB NumPy 배열, BGR 반환)"""
    img = image.copy()
    h, w = img.shape[:2]
    # BGR로 변환 (cv2 표준)
    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)

    # 랜드마크 좌표 (정규화 0~1 → 픽셀)
    points = []
    for lm in hand_landmarks:
        px = int(lm.x * w)
        py = int(lm.y * h)
        points.append((px, py))

    # 연결선
    for (a, b) in connections:
        cv2.line(img, points[a], points[b], connection_color, thickness)

    # 랜드마크 점
    for i, (px, py) in enumerate(points):
        cv2.circle(img, (px, py), radius, landmark_color, -1)
        cv2.putText(img, str(i), (px + 4, py - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

    return img  # RGB 반환 (matplotlib용)

print('✔ draw_hand_landmarks() 함수 정의 완료')


In [ ]:
# 합성 결과로 시각화 함수 테스트 (가짜 랜드마크)
def make_fake_landmarks():
    """데모용 가짜 21개 랜드마크 (정규화 좌표)"""
    # 손 모양 대략 배치
    pts = [
        (0.5, 0.9),   # 0 wrist
        (0.40, 0.80), (0.32, 0.70), (0.28, 0.62), (0.25, 0.55),  # 1-4 thumb
        (0.45, 0.55), (0.45, 0.42), (0.45, 0.34), (0.45, 0.28),  # 5-8 index
        (0.55, 0.53), (0.56, 0.38), (0.56, 0.30), (0.56, 0.24),  # 9-12 middle
        (0.65, 0.55), (0.67, 0.42), (0.68, 0.34), (0.69, 0.28),  # 13-16 ring
        (0.75, 0.60), (0.78, 0.50), (0.80, 0.44), (0.82, 0.40),  # 17-20 pinky
    ]
    class Lm:
        def __init__(self, x, y, z=0.0):
            self.x, self.y, self.z = x, y, z
    return [Lm(x, y) for (x, y) in pts]

fake = make_fake_landmarks()
demo_img = np.full((480, 640, 3), 220, dtype=np.uint8)
annotated = draw_hand_landmarks(demo_img, fake)

plt.figure(figsize=(8, 6))
plt.imshow(annotated)
plt.title('21개 손 랜드마크 (데모)')
plt.axis('off')
plt.tight_layout()
plt.show()
print('✔ 시각화 완료 (실제 detect 결과에도 동일 함수 사용 가능)')


## 6. handedness (왼손 / 오른손)

`result.handedness`는 각 손에 대해 `Category` 객체 리스트를 반환합니다.
- `category_name`: `"Left"` 또는 `"Right"`
- `score`: 신뢰도 (0~1)

> ⚠️ handedness는 **이미지에 찍힌 손 기준**이며, 카메라 거울 모드 여부에 따라
> 사용자 체감과 다를 수 있습니다. 셀카(전면 카메라)에서는 좌우가 반전되므로
> 필요하면 `Left ↔ Right`를 직접 바꿔 표시하세요.


In [ ]:
def print_handedness(result):
    """검출 결과에서 handedness 정보 출력"""
    if not result.handedness:
        print('검출된 손이 없습니다.')
        return
    for i, hand_cats in enumerate(result.handedness):
        cat = hand_cats[0]  # 첫 번째 카테고리 사용
        label = cat.category_name
        score = cat.score
        print(f'손 {i}: {label} (신뢰도: {score:.3f})')

# 합성 이미지 결과로 데모 (손 없음 → 빈)
print('--- 합성 이미지 결과 ---')
print_handedness(result)

# 실제 detect 결과가 있을 때의 사용 예시
print('\n--- 사용 예시 (실제 결과에서) ---')
print('# for i, hand in enumerate(result.hand_landmarks):')
print('#     cats = result.handedness[i]')
print('#     label = cats[0].category_name  # "Left" / "Right"')
print('#     score = cats[0].score')


## 7. 웹캠 실시간 검출 — LIVE_STREAM + 콜백

실시간 웹캠 처리에는 `RunningMode.LIVE_STREAM`을 사용합니다.
비동기 추론을 위해 **콜백 함수**를 등록하고 `detect_async()`를 호출합니다.

```
┌──────────────────────────────────────────────────────────────┐
│  LIVE_STREAM 워크플로우                                       │
│                                                              │
│  1) result_callback(result, output_image, timestamp_ms) 정의 │
│  2) HandLandmarkerOptions(running_mode=LIVE_STREAM,          │
│                          result_callback=callback)           │
│  3) 루프: detect_async(mp_image, timestamp_ms)               │
│  4) 콜백 내부에서 최신 결과를 전역 변수에 저장                │
│  5) 메인 루프에서 최신 결과를 화면에 그림                     │
└──────────────────────────────────────────────────────────────┘
```

> 콜백은 별도 스레드에서 실행되므로, **전역 변수에 최신 결과만 저장**하고
> 실제 그리기는 메인 스레드에서 수행하는 패턴이 안전합니다.


In [ ]:
# LIVE_STREAM용 검출기 생성 (콜백 포함)
import time

# 최신 결과를 저장할 전역 변수
latest_result = None
latest_timestamp = 0

def result_callback(result, output_image, timestamp_ms):
    """LIVE_STREAM 콜백 — 별도 스레드에서 호출됨"""
    global latest_result, latest_timestamp
    latest_result = result
    latest_timestamp = timestamp_ms

# LIVE_STREAM 옵션
stream_options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.LIVE_STREAM,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=result_callback,
)

# 기존 IMAGE 모드 검출기 닫기
detector.close()

# LIVE_STREAM 검출기 생성
stream_detector = vision.HandLandmarker.create_from_options(stream_options)
print('✔ LIVE_STREAM HandLandmarker 생성 완료')


In [ ]:
# 웹캠 실시간 손 랜드마크 검출 (LIVE_STREAM)
# ⚠️ 웹캠이 연결된 환경에서만 실행하세요. 헤드리스 환경에서는 건너뛰세요.

def run_webcam_hand_landmark():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('❌ 웹캠을 열 수 없습니다. (index 0)')
        return

    print('웹캠 시작 — ESC 키로 종료')
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        timestamp_ms = int(time.time() * 1000)

        # BGR → RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

        # 비동기 추론 (콜백이 latest_result를 갱신)
        stream_detector.detect_async(mp_image, timestamp_ms)

        # 최신 결과가 있으면 그리기 (메인 스레드)
        if latest_result is not None and len(latest_result.hand_landmarks) > 0:
            for hand in latest_result.hand_landmarks:
                annotated = draw_hand_landmarks(frame_rgb, hand)
                frame_rgb = annotated
            # handedness 표시
            for i, hand_cats in enumerate(latest_result.handedness):
                label = hand_cats[0].category_name
                cv2.putText(frame_rgb, label, (10, 30 + i * 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        # FPS 표시
        cv2.putText(frame_rgb, f'frame: {frame_idx}', (10, 470),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 128, 0), 1)

        # 화면 출력 (RGB → BGR)
        cv2.imshow('Hand Landmark (LIVE_STREAM)', cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()
    print('웹캠 종료')

# 실행 (주석 해제하여 사용)
# run_webcam_hand_landmark()
print('✔ run_webcam_hand_landmark() 함수 정의 완료')
print('  실행하려면 위 주석을 해제하세요: run_webcam_hand_landmark()')


## 8. 응용: 손가락 개수 세기 (Finger Counting)

21개 랜드마크를 이용하면 **펴진 손가락 개수**를 셀 수 있습니다.

**판정 원리**
- 4개 손가락(검지/중지/약지/새끼): **TIP(끝)이 PIP(첫 마디)보다 위**에 있으면 펴짐
  - (이미지 좌표에서 y가 작을수록 위)
- 엄지: **TIP(4)가 IP(3)보다 손목 반대쪽**에 있으면 펴짐 (좌우 위치로 판정)

```
┌──────────────────────────────────────────────────────┐
│  손가락별 랜드마크 인덱스                             │
│                                                      │
│  엄지:   TIP=4,  IP=3,  MCP=2                        │
│  검지:   TIP=8,  PIP=6, MCP=5                        │
│  중지:   TIP=12, PIP=10, MCP=9                       │
│  약지:   TIP=16, PIP=14, MCP=13                      │
│  새끼:   TIP=20, PIP=18, MCP=17                      │
│  손목:   0                                           │
└──────────────────────────────────────────────────────┘
```


In [ ]:
def count_fingers(hand_landmarks, handedness_label='Right'):
    """
    펴진 손가락 개수 반환 (0~5)
    hand_landmarks: 21개 NormalizedLandmark 리스트 (x, y, z)
    handedness_label: 'Left' 또는 'Right' (엄지 판정 방향 보정용)
    """
    # 손가락별 (TIP, PIP) 인덱스 — 엄지 제외 4개
    finger_tip_pip = [
        (8, 6),   # 검지
        (12, 10), # 중지
        (16, 14), # 약지
        (20, 18), # 새끼
    ]

    count = 0

    # 4개 손가락: TIP.y < PIP.y 이면 펴짐 (y는 위가 작음)
    for tip, pip in finger_tip_pip:
        if hand_landmarks[tip].y < hand_landmarks[pip].y:
            count += 1

    # 엄지: TIP(4)가 IP(3)보다 좌/우로 더 멀리 있으면 펴짐
    # handedness에 따라 기준 방향이 반대
    thumb_tip = hand_landmarks[4]
    thumb_ip = hand_landmarks[3]
    thumb_mcp = hand_landmarks[2]
    wrist = hand_landmarks[0]

    # 엄지가 펴졌는지: TIP와 IP의 x 차이로 판정 (손목 기준)
    # 단순히 TIP.x가 IP.x보다 손목에서 멀리 있으면 펴짐으로 간주
    if handedness_label == 'Right':
        if thumb_tip.x < thumb_ip.x:  # 오른손: 엄지가 왼쪽으로 뻗으면 펴짐
            count += 1
    else:  # Left
        if thumb_tip.x > thumb_ip.x:  # 왼손: 엄지가 오른쪽으로 뻗으면 펴짐
            count += 1

    return count

# 데모: 가짜 랜드마크로 테스트
fake = make_fake_landmarks()
print(f'펴진 손가락 개수 (데모, Right): {count_fingers(fake, "Right")}')

# 모든 손가락을 쭉 편 가짜 데이터로 다시 테스트 (엄지도 펴짐)
fake_open = make_fake_landmarks()
fake_open[4] = type(fake_open[4])(0.15, 0.50)  # 엄지 끝을 더 왼쪽으로
print(f'펴진 손가락 개수 (엄지 펴짐, Right): {count_fingers(fake_open, "Right")}')


In [ ]:
# 손가락 개수 + 랜드마크를 함께 그리는 함수
def draw_hand_with_count(image, hand_landmarks, handedness_label='Right'):
    """랜드마크 시각화 + 손가락 개수 표시"""
    annotated = draw_hand_landmarks(image, hand_landmarks)
    count = count_fingers(hand_landmarks, handedness_label)
    text = f'Fingers: {count}  ({handedness_label})'
    cv2.putText(annotated, text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 200), 2)
    return annotated, count

# 데모
demo_img = np.full((480, 640, 3), 220, dtype=np.uint8)
fake = make_fake_landmarks()
annotated, count = draw_hand_with_count(demo_img, fake, 'Right')

plt.figure(figsize=(8, 6))
plt.imshow(annotated)
plt.title(f'Finger Count = {count}')
plt.axis('off')
plt.tight_layout()
plt.show()


### 웹캠 실시간 손가락 개수 세기 (LIVE_STREAM 응용)

위 `run_webcam_hand_landmark()`에서 `draw_hand_with_count()`를 사용하면
실시간으로 손가락 개수를 화면에 표시할 수 있습니다.


In [ ]:
# 웹캠 + 손가락 카운팅 (LIVE_STREAM)
def run_webcam_finger_count():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('❌ 웹캠을 열 수 없습니다.')
        return
    print('웹캠 시작 — ESC 종료')
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        timestamp_ms = int(time.time() * 1000)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        stream_detector.detect_async(mp_image, timestamp_ms)

        if latest_result is not None and len(latest_result.hand_landmarks) > 0:
            for i, hand in enumerate(latest_result.hand_landmarks):
                label = latest_result.handedness[i][0].category_name
                frame_rgb, count = draw_hand_with_count(frame_rgb, hand, label)

        cv2.imshow('Finger Count', cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
        if cv2.waitKey(1) & 0xFF == 27:
            break
    cap.release()
    cv2.destroyAllWindows()
    print('종료')

# run_webcam_finger_count()
print('✔ run_webcam_finger_count() 함수 정의 완료')


---
## 🎯 정리

| 개념 | 설명 |
|------|------|
| **HandLandmarker** | 손의 21개 랜드마크를 검출하는 Tasks API 클래스 |
| **모델 파일** | `models/hand_landmarker.task` |
| **BaseOptions** | `model_asset_path`로 모델 경로 지정 |
| **RunningMode.IMAGE** | 정적 이미지 — `detect(mp_image)` 호출 |
| **RunningMode.LIVE_STREAM** | 실시간 — `detect_async(mp_image, timestamp_ms)` + 콜백 |
| **mp.Image** | `mp.Image(mp.ImageFormat.SRGB, np_array)` 로 NumPy 변환 |
| **21개 랜드마크** | 손목(1) + 손가락 5개 × 4관절(20) = 21 |
| **손가락 관절** | MCP(밑동) → PIP(첫 마디) → DIP(둘째 마디) → TIP(끝) |
| **handedness** | `result.handedness[i][0].category_name` → "Left"/"Right" |
| **좌표계** | x, y는 0~1 정규화, z는 손목 기준 상대 깊이 |
| **손가락 카운팅** | TIP.y < PIP.y → 펴짐 / 엄지는 x 방향으로 판정 |
| **result_callback** | LIVE_STREAM에서 결과를 받는 콜백 (별도 스레드) |

| 단계 | 핵심 코드 |
|------|-----------|
| 옵션 설정 | `vision.HandLandmarkerOptions(base_options=..., running_mode=IMAGE, num_hands=2)` |
| 검출기 생성 | `vision.HandLandmarker.create_from_options(options)` |
| 정적 검출 | `result = detector.detect(mp_image)` |
| 실시간 검출 | `stream_detector.detect_async(mp_image, timestamp_ms)` |
| 결과 접근 | `result.hand_landmarks[hand_idx][lm_idx].x/.y/.z` |
| handedness | `result.handedness[hand_idx][0].category_name` |


---
## 🎯 연습 문제

1. **랜드마크 출력**: `detect()` 결과에서 검지 끝(INDEX_TIP, 8번)과 엄지 끝(THUMB_TIP, 4번)의
   `(x, y, z)` 좌표를 출력하는 코드를 작성하세요. 두 점 사이의 거리도 계산해 보세요.

2. **handedness 보정**: 셀카 모드(전면 카메라)에서는 좌우가 반전됩니다.
   `result.handedness`의 `"Left"`/`"Right"`를 사용자 체감 기준으로 바꾸어 표시하는
   함수 `get_user_handedness(result, mirror=True)`를 작성하세요.

3. **제스처 인식**: 손가락 개수를 이용해 간단한 제스처를 분류해 보세요.
   - 1개: "One" / 2개: "Two" / 3개: "Three" / 4개: "Four" / 5개: "Open"
   - 0개: "Fist"
   `classify_gesture(count)` 함수를 작성하고 웹캠에 표시해 보세요.

4. **VIDEO 모드**: `RunningMode.VIDEO` 모드로 동영상 파일에서 손 랜드마크를 추출하는
   코드를 작성하세요. `detect_for_video(mp_image, timestamp_ms)`를 사용하며
   콜백 없이 동기적으로 결과를 반환받습니다. (힌트: `cv2.VideoCapture('video.mp4')`)
